In [6]:
METHODS = ['phi']

In [7]:
import numpy as np
import pandas as pd
from fancyimpute import SoftImpute

from langrank.replace_distances import replace_in_memory
from langrank.dep.dep import dep_in_memory
from langrank.el.el import el_in_memory
from langrank.mt.mt import mt_in_memory
from langrank.pos.pos import pos_in_memory

from config import IMPUTATION   # whether to use imputed values

FEATURE_TYPES = ['GENETIC','SYNTACTIC','FEATURAL','PHONOLOGICAL','INVENTORY','GEOGRAPHIC']
TASKS = ['mt', 'dep', 'el', 'pos']  # Langrank tasks to evaluate on

# Initialize results dictionary
results: dict[str, dict[str, list[float]]] = {method: {task: [] for task in TASKS} for method in METHODS}
for method in METHODS:
    results[method]['num_features'] = []

# Imputation after UFSACO
The following cell imputes the data using SoftImpute on the results from UFSACO (unsupervised feature selection based on ant colony optimization). Only runs if IMPUTATION is set.

In [8]:
if IMPUTATION:
    for hueristic in METHODS:
        for num_features in range(100, 701, 100):
            df: pd.DataFrame = pd.read_csv(f'selection_result/ant_{hueristic}_{num_features}.csv', index_col=0)

            df_np = df.to_numpy()
            df_np = np.where(df_np == -1, np.nan, df_np)

            imputer = SoftImpute(max_iters=400,  max_value=1, min_value=0, init_fill_method="mean")
            imputed_values = imputer.fit_transform(df_np)

            df_imputed = pd.DataFrame(imputed_values, columns=df.columns, index=df.index)
            df_imputed.to_csv(f'selection_result/imputed_ant_{hueristic}_{num_features}.csv')

# Evalulate feature selected subset on Langrank

In [9]:
for method in METHODS:
    for i in range(100, 701, 100):
        results[method]['num_features'].append(i)

        path = f'selection_result/imputed_ant_{method}_{i}.csv' if IMPUTATION else f'selection_result/ant_{method}_{i}.csv'
        df = pd.read_csv(path, index_col=0)

        dep_df, el_df, mt_df, pos_df = replace_in_memory(df)
        dep_ndcg: float = dep_in_memory(dep_df, FEATURE_TYPES)
        el_ndcg: float = el_in_memory(el_df, FEATURE_TYPES)
        mt_ndcg: float = mt_in_memory(mt_df, FEATURE_TYPES)
        pos_ndcg: float = pos_in_memory(pos_df, FEATURE_TYPES)

        results[method]['dep'].append(dep_ndcg)
        results[method]['el'].append(el_ndcg)
        results[method]['mt'].append(mt_ndcg)
        results[method]['pos'].append(pos_ndcg)

        print(f'Finished {method} with {i} features')

    results_df = pd.DataFrame(results[method])
    results_df.set_index('num_features', inplace=True)

    save = f'eval_result/imputed_ant_{method}_results.csv' if IMPUTATION else f'eval_result/ant_{method}_results.csv'
    results_df.to_csv(save)

[LightGBM] [Warning] min_data_in_leaf is set=5, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=5
[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
[LightGBM] [Warning] min_data_in_leaf is set=5, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=5
[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
[LightGBM] [Warning] min_data_in_leaf is set=5, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=5
[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
[LightGBM] [Warning] min_data_in_leaf is set=5, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=5
[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
[LightGBM] [Warning] min

# Evalulate baseline results on Langrank

In [10]:
path = 'data/URIELPlus_Union_Imputed.csv' if IMPUTATION else 'data/URIELPlus_Union.csv'
df = pd.read_csv(path, index_col=0)

dep_df, el_df, mt_df, pos_df = replace_in_memory(df)
dep_ndcg: float = dep_in_memory(dep_df, FEATURE_TYPES)
el_ndcg: float = el_in_memory(el_df, FEATURE_TYPES)
mt_ndcg: float = mt_in_memory(mt_df, FEATURE_TYPES)
pos_ndcg: float = pos_in_memory(pos_df, FEATURE_TYPES)

results['baseline'] = {
    'dep': [dep_ndcg],
    'el': [el_ndcg],
    'mt': [mt_ndcg],
    'pos': [pos_ndcg]
}

# Save baseline results to CSV
baseline_df = pd.DataFrame(results['baseline'])

save = 'eval_result/baseline_results_imputed.csv' if IMPUTATION else 'eval_result/baseline_results.csv'
baseline_df.to_csv(save, index=False)

[LightGBM] [Warning] min_data_in_leaf is set=5, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=5
[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
[LightGBM] [Warning] min_data_in_leaf is set=5, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=5
[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
[LightGBM] [Warning] min_data_in_leaf is set=5, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=5
[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
[LightGBM] [Warning] min_data_in_leaf is set=5, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=5
[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
[LightGBM] [Warning] min